# BayesNF — final held-out evaluation

Single VI run. Train: all non-holdout stations
(`bayesnf_final_train.parquet`). Test: 492 held-out stations
(`bayesnf_holdout_test.parquet`). Period: 2010-2023.

Artifacts:

```
s3://thesis-data-ismaktam/bayesnf/runs/vi__final__WY_h1_10__ffrk_full/
    config.json   metrics.json   preds.parquet   losses.npy   model.pkl
```


## 1. Environment + imports


In [ ]:
import os
# JAX env vars must be set before the first `import jax`.
os.environ.setdefault('JAX_LOG_COMPILES', '0')
os.environ.setdefault('XLA_PYTHON_CLIENT_MEM_FRACTION', '0.90')

# Speedup flags (validated in speedup/speedup.ipynb): Triton GEMM,
# latency-hiding scheduler, async collectives.
os.environ['XLA_FLAGS'] = (
    '--xla_gpu_enable_triton_gemm=true '
    '--xla_gpu_enable_latency_hiding_scheduler=true '
    '--xla_gpu_enable_async_collectives=true'
)

import sys
import gc
import time
import json
import pickle
import threading
from pathlib import Path

import numpy as np
import pandas as pd

import jax
import jax.numpy as jnp
import bayesnf
from bayesnf.spatiotemporal import (
    BayesianNeuralFieldVI,
    BayesianNeuralFieldMAP,
)

# bf16 mixed precision: enables tensor cores on A100. Smoke A/B vs tf32
# matched loss to the 5th significant digit and saves ~1/2 of activation
# memory, which lets us keep batch=131072 on long periods (>10 years).
jax.config.update('jax_default_matmul_precision', 'bfloat16')
PRECISION_NOTE = 'bfloat16 (A100 tensor cores)'

# Repo root: notebooks/05_bayesnf/final/<this>.ipynb -> ../../..
ROOT = Path('../../..').resolve()
sys.path.insert(0, str(ROOT / 'src'))
os.chdir(ROOT)

print(f'cwd      : {os.getcwd()}')
print(f'bayesnf  : {getattr(bayesnf, "__version__", "n/a")}')
print(f'precision: {PRECISION_NOTE}')


## 2. GPU diagnostics


In [ ]:
import subprocess

backend = jax.default_backend()
devices = jax.devices()
print(f'JAX backend : {backend}')
print(f'JAX devices : {devices}')
print(f'n local     : {jax.local_device_count()}')

if backend != 'gpu':
    print('\nWARNING: JAX is on CPU — this notebook is intended for GPU.')

try:
    smi = subprocess.run(
        ['nvidia-smi', '--query-gpu=name,memory.used,memory.total',
         '--format=csv,noheader'],
        capture_output=True, text=True, timeout=5,
    )
    print('\nnvidia-smi:')
    print(smi.stdout)
except Exception as e:
    print(f'nvidia-smi unavailable: {e}')


## 3. Data and target config


In [ ]:
LIKELIHOOD    = 'ZINB'
TARGET_COL    = 'rainfall_int'
NEEDS_RESCALE = True
PRECIP_SCALE  = 10

FOLD       = -1
YEAR_START = 2010
YEAR_END   = 2023

LOCAL_RUNS = Path('results/bayesnf/runs')
LOCAL_RUNS.mkdir(parents=True, exist_ok=True)

S3_BUCKET    = 'thesis-data-ismaktam'
S3_RUNS_ROOT = 'bayesnf/runs'

print(f'likelihood : {LIKELIHOOD}  (target={TARGET_COL}, rescale={NEEDS_RESCALE})')
print(f'years      : {YEAR_START}-{YEAR_END}')
print(f'local runs : {LOCAL_RUNS}')
print(f's3 prefix  : s3://{S3_BUCKET}/{S3_RUNS_ROOT}/<name>/')


## 4. Data load

Train: `bayesnf_final_train.parquet` (all non-holdout stations, 1961-2023,
filtered to `YEAR_START..YEAR_END`).
Test: `bayesnf_holdout_test.parquet` (492 held-out stations, same period).

Both files share the same 39-column schema. No `station_id` overlap is
allowed between train and test.


In [ ]:
DATA_S3_DIR = 'bayesnf/data'
DATA_DIR    = Path('results/bayesnf/data')
DATA_DIR.mkdir(parents=True, exist_ok=True)

train_path = DATA_DIR / 'bayesnf_final_train.parquet'
test_path  = DATA_DIR / 'bayesnf_holdout_test.parquet'

def _ensure_local(local_path: Path, s3_key: str) -> None:
    if local_path.exists():
        print(f'  cache hit: {local_path.name} ({local_path.stat().st_size/1e6:.0f} MB)')
        return
    import boto3
    print(f'  downloading s3://{S3_BUCKET}/{s3_key} ...')
    boto3.client('s3').download_file(S3_BUCKET, s3_key, str(local_path))
    print(f'  downloaded {local_path.name} ({local_path.stat().st_size/1e6:.0f} MB)')

_ensure_local(train_path, f'{DATA_S3_DIR}/bayesnf_final_train.parquet')
_ensure_local(test_path,  f'{DATA_S3_DIR}/bayesnf_holdout_test.parquet')

date_filter = [
    ('datetime', '>=', pd.Timestamp(f'{YEAR_START}-01-01')),
    ('datetime', '<=', pd.Timestamp(f'{YEAR_END}-12-31')),
]

ALL_GEO_COLS = [
    'datetime', 'latitude', 'longitude', 'x_proj', 'y_proj', 'elevation_m',
    'idw', 'gos',
    'svd_05', 'svd_06', 'svd_07', 'svd_08', 'svd_09',
    'bulk_density', 'clay', 'sand', 'silt', 'soc', 'water_10kpa',
]
KEEP_COLS = ALL_GEO_COLS + ['rainfall', 'rainfall_int', 'station_id']

df_train = pd.read_parquet(train_path, filters=date_filter)[KEEP_COLS].copy()
df_test  = pd.read_parquet(test_path,  filters=date_filter)[KEEP_COLS].copy()
df_train['datetime'] = pd.to_datetime(df_train['datetime'])
df_test ['datetime'] = pd.to_datetime(df_test ['datetime'])

assert df_train[ALL_GEO_COLS].isna().sum().sum() == 0
assert df_test [ALL_GEO_COLS].isna().sum().sum() == 0
assert 'rainfall_int' in df_train.columns

train_ids = set(df_train['station_id'].unique())
test_ids  = set(df_test ['station_id'].unique())
overlap = train_ids & test_ids
assert not overlap, f'station_id overlap: {len(overlap)} ids'

print(f'train : {len(df_train):>10,} rows / {len(train_ids):>4} stations')
print(f'test  : {len(df_test):>10,} rows / {len(test_ids):>4} stations (holdout)')
print(f'dates : {df_train["datetime"].min().date()} -> {df_train["datetime"].max().date()}')
print(f'%zero : {100*(df_train["rainfall_int"]==0).mean():.1f}% (rainfall_int)')


## 5. JIT-cache patch for `predict`

Same patch as in time_seasonality / geo_features. Without it `predict_bnf`
recompiles the closure on every chunk.


In [ ]:
import bayesnf.inference as bnf_inference
import bayesnf.models as bnf_models
import bayesnf.spatiotemporal as bnf_st
from tensorflow_probability.substrates import jax as _tfp_jax

_tfd = _tfp_jax.distributions
_LD  = bnf_models.LikelihoodDist

# --- bounded likelihood (anti-blowup for NB / ZINB) ---------------------------
# In bayesnf NB/ZINB use mean = softplus(MLP(x)), shape = softplus(p1).
# MLP output is unbounded -> on test rows with extreme features `mean`
# blows up to ~10^6 mm. Fix: a thin tanh wrapper around the raw params.
#
#  raw_pred  -> PRED_CAP * tanh(raw_pred / PRED_CAP)   (soft clip; in the
#               linear regime |raw_pred| << PRED_CAP it does not hurt training)
#  params[1] -> SHAPE_CAP * tanh(params[1] / SHAPE_CAP) -> shape in ~[2.5e-3, 6]
#               -> total_count = 1/shape in ~[0.17, 400].
#
# Target range of mean in rainfall_int units: <= 10*200 mm = 2000.
# PRED_CAP = 4000 (2x headroom) -> softplus(+/-4000) ~ {~0, 4000}.
PRED_CAP  = 4000.0
SHAPE_CAP = 6.0

_orig_make_likelihood = bnf_models.make_likelihood_model


def make_likelihood_bounded(params, x, mlp, mlp_template, distribution):
    if _LD(distribution) == _LD.NORMAL:
        return _orig_make_likelihood(params, x, mlp, mlp_template, distribution)

    treedef    = jax.tree_util.tree_structure(mlp_template)
    mlp_params = jax.tree_util.tree_unflatten(treedef, params[3:])
    raw_pred   = mlp.apply(mlp_params, x)
    bounded_pred = PRED_CAP * jnp.tanh(raw_pred / PRED_CAP)
    mean       = jax.nn.softplus(bounded_pred)

    bounded_p1 = SHAPE_CAP * jnp.tanh(params[1] / SHAPE_CAP)
    shape      = jax.nn.softplus(bounded_p1)
    logits     = -jnp.log(shape) - jnp.log(mean)

    if _LD(distribution) == _LD.NB:
        return _tfd.Independent(
            _tfd.NegativeBinomial(total_count=1.0 / shape, logits=logits), 1)
    elif _LD(distribution) == _LD.ZINB:
        inflated_loc_probs = 1.0 / (1.0 + jnp.exp(-params[2]))
        return _tfd.Independent(
            _tfd.ZeroInflatedNegativeBinomial(
                total_count=1.0 / shape, logits=logits,
                inflated_loc_probs=inflated_loc_probs * jnp.ones(shape=mean.shape)), 1)
    raise AssertionError(f'unknown distribution: {distribution}')


bnf_models.make_likelihood_model = make_likelihood_bounded
print(f'bounded likelihood: mean <= ~{PRED_CAP:.0f} (rainfall_int units = '
      f'{PRED_CAP/PRECIP_SCALE:.0f} mm/day),  total_count in [{1/jax.nn.softplus(SHAPE_CAP):.3f}, '
      f'{1/jax.nn.softplus(-SHAPE_CAP):.1f}]')


# --- predict JIT-cache (same patch as in time_seasonality / geo_features) ----
def _patched_predict(self, table, quantiles=(0.5,), approximate_quantiles=False):
    test_data = self.data_handler.get_test(table)
    distribution = bnf_models.LikelihoodDist(self.observation_model)

    if getattr(self, '_cached_forecast_inner', None) is None:
        model_args = self._model_args(test_data.shape)
        fn = bnf_inference._make_forecast_inner(model_args, distribution)
        for _ in range(self._ensemble_dims - 1):
            fn = jax.vmap(fn, in_axes=(0, None))
        fn = jax.pmap(fn, in_axes=(0, None))
        self._cached_forecast_inner = fn

    forecast_inner = self._cached_forecast_inner
    axis = tuple(range(self._ensemble_dims))

    forecast_params = bnf_inference.forecast_parameters_batched(
        test_data, self.params_, distribution, forecast_inner,
    )

    if distribution == bnf_models.LikelihoodDist.NORMAL:
        means, scales = forecast_params
        forecast_means = means
        forecast_quantiles = bnf_inference._get_percentile_normal(
            forecast_means, scales, quantiles, axis=axis,
            approximate=approximate_quantiles,
        )
    elif distribution in (bnf_models.LikelihoodDist.NB, bnf_models.LikelihoodDist.ZINB):
        obs_d = bnf_inference._build_observation_distribution(distribution, forecast_params)
        forecast_means = obs_d.mean()
        forecast_quantiles = jax.vmap(
            lambda q: bnf_inference._get_nb_quantiles_root(obs_d, q, ensemble_axes=axis)
        )(jnp.array(quantiles))
    else:
        raise ValueError(f'Unknown distribution: {distribution}')

    return forecast_means, forecast_quantiles


bnf_st.BayesianNeuralFieldEstimator.predict = _patched_predict
print('patched BayesianNeuralFieldEstimator.predict')


## 5b. Multi-GPU predict — larger inner batchsize

`bayesnf.inference.forecast_parameters_batched` processes the test data
**in chunks of 1024 rows** in a Python loop. With PRED_CHUNK = 500_000
this is ~488 sequential pmap calls per chunk. On A100/A40 a forward
pass over 1024 rows is too small to saturate the GPUs.

Fix: same pmap(vmap(vmap)) as in baseline, but with
`batchsize >= 16_384`. Less Python overhead, higher GPU utilisation,
predict is **5–10x** faster depending on the instance.

`N_DEVICES` is auto-detected via `jax.local_device_count()` so the same
notebook runs on 1x/4x/8x GPU without edits.

> Sanity: the result is bitwise equivalent to baseline (same pmap structure,
> only inner batch size changes). To verify, run one small fold with both
> patches and `assert np.allclose(...)`.


In [ ]:
import jax
import jax.numpy as jnp
import bayesnf.inference as bnf_inference
import bayesnf.models as bnf_models
import bayesnf.spatiotemporal as bnf_st

N_DEVICES = jax.local_device_count()

# Per-device inner batch. Memory peak per device:
#   per_device_batch × ensemble_size × MLP_width × n_layers × dtype_bytes
# For ensemble_size=16, width=256, depth=2, float32 ≈ 130 KB/row → 16k rows ≈ 2GB
# Comfortable for 40-80GB cards; cap conservatively for smaller GPUs.
PER_DEVICE_BATCH = 16_384
PREDICT_BATCHSIZE = max(PER_DEVICE_BATCH * N_DEVICES, 16_384)

print(f'[predict-tune] N_DEVICES         = {N_DEVICES}')
print(f'[predict-tune] PER_DEVICE_BATCH  = {PER_DEVICE_BATCH:,}')
print(f'[predict-tune] PREDICT_BATCHSIZE = {PREDICT_BATCHSIZE:,}  '
      f'(was 1024 in baseline → ~{PREDICT_BATCHSIZE//1024}× larger)')


def _patched_predict_v2(self, table, quantiles=(0.5,), approximate_quantiles=False):
    test_data = self.data_handler.get_test(table)
    distribution = bnf_models.LikelihoodDist(self.observation_model)

    if getattr(self, '_cached_forecast_inner', None) is None:
        model_args = self._model_args(test_data.shape)
        fn = bnf_inference._make_forecast_inner(model_args, distribution)
        for _ in range(self._ensemble_dims - 1):
            fn = jax.vmap(fn, in_axes=(0, None))
        fn = jax.pmap(fn, in_axes=(0, None))
        self._cached_forecast_inner = fn

    forecast_inner = self._cached_forecast_inner
    axis = tuple(range(self._ensemble_dims))

    forecast_params = bnf_inference.forecast_parameters_batched(
        test_data, self.params_, distribution, forecast_inner,
        batchsize=PREDICT_BATCHSIZE,
    )

    if distribution == bnf_models.LikelihoodDist.NORMAL:
        means, scales = forecast_params
        forecast_means = means
        forecast_quantiles = bnf_inference._get_percentile_normal(
            forecast_means, scales, quantiles, axis=axis,
            approximate=approximate_quantiles,
        )
    elif distribution in (bnf_models.LikelihoodDist.NB, bnf_models.LikelihoodDist.ZINB):
        obs_d = bnf_inference._build_observation_distribution(distribution, forecast_params)
        forecast_means = obs_d.mean()
        forecast_quantiles = jax.vmap(
            lambda q: bnf_inference._get_nb_quantiles_root(obs_d, q, ensemble_axes=axis)
        )(jnp.array(quantiles))
    else:
        raise ValueError(f'Unknown distribution: {distribution}')

    return forecast_means, forecast_quantiles


bnf_st.BayesianNeuralFieldEstimator.predict = _patched_predict_v2
print(f'✓ predict re-patched with batchsize={PREDICT_BATCHSIZE:,} '
      f'across {N_DEVICES} device(s)')


## 6. Helpers: metrics, save, S3 upload, train_eval_vi / train_eval_map

One block. Runs once; the two training cells below just call
`train_eval_vi(name=..., **overrides)` or `train_eval_map(...)`.

**Metrics.**

| Group | Slice | Metrics |
|--------|-------|---------|
| Global | all test rows | RMSE, MAE, bias, CRPS, cov90, cov80 |
| Wet    | `y >= 0.5 mm` | RMSE_wet, MAE_wet, CRPS_wet, cov90_wet, cov80_wet |
| Dry    | `y < 0.5 mm` | MAE_dry, RMSE_dry, bias_dry |
| Detector | wet-classifier with threshold `predicted >= 0.4 mm`, observed `>= 0.5 mm` | TP/FP/TN/FN, accuracy, precision, recall, F1, Brier |


In [ ]:
from properscoring import crps_ensemble
import boto3

# --- thresholds ---
WET_THRESHOLD_MM   = 0.5   # y_true wet iff y >= 0.5 mm
PRED_DRY_THRESHOLD = 0.4   # model predicts dry iff mean_mm < 0.4

# --- quantiles ---
QUANTILES = (0.05, 0.1, 0.2, 0.3, 0.4, 0.50, 0.6, 0.7, 0.8, 0.9, 0.95)
QUANTILE_LABELS = [5, 10, 20, 30, 40, 50, 60, 70, 80, 90, 95]
Q05_I = QUANTILE_LABELS.index(5)
Q10_I = QUANTILE_LABELS.index(10)
Q50_I = QUANTILE_LABELS.index(50)
Q90_I = QUANTILE_LABELS.index(90)
Q95_I = QUANTILE_LABELS.index(95)

PRED_CHUNK = 500_000

# --- defaults (geo features + seasonality) ---
GEO_DEFAULTS = dict(
    feature_cols=['datetime', 'latitude', 'longitude', 'elevation_m',
                  'idw', 'gos',
                  'svd_05', 'svd_06', 'svd_07', 'svd_08', 'svd_09',
                  'bulk_density', 'clay', 'sand', 'silt', 'soc', 'water_10kpa'],
    interactions=[(0, 1), (0, 2), (0, 3), (0, 4), (0, 5),
                  (1, 2), (1, 3), (2, 3)],
)
SEAS_DEFAULT = dict(periods=['W', 'Y'], harmonics=[1, 10])


# ----------------------------------------------------------------------
def _metrics(y_true, mean_mm, q_arr):
    """Returns dict with global / wet / dry / detector groups."""
    lo90 = q_arr[:, Q05_I]; hi90 = q_arr[:, Q95_I]
    lo80 = q_arr[:, Q10_I]; hi80 = q_arr[:, Q90_I]

    obs_wet  = y_true  >= WET_THRESHOLD_MM
    pred_wet = mean_mm >= PRED_DRY_THRESHOLD
    tp = int(( obs_wet &  pred_wet).sum())
    tn = int((~obs_wet & ~pred_wet).sum())
    fp = int((~obs_wet &  pred_wet).sum())
    fn = int(( obs_wet & ~pred_wet).sum())
    n  = len(y_true)
    eps = 1e-12

    accuracy  = (tp + tn) / max(n, 1)
    precision = tp / max(tp + fp, 1)
    recall    = tp / max(tp + fn, 1)
    f1        = 2 * precision * recall / max(precision + recall, eps)
    brier     = float(np.mean((pred_wet.astype(np.float32) - obs_wet.astype(np.float32))**2))

    out = {
        # global
        'rmse'    : float(np.sqrt(np.mean((mean_mm - y_true) ** 2))),
        'mae'     : float(np.mean(np.abs(mean_mm - y_true))),
        'bias'    : float(np.mean(mean_mm - y_true)),
        'crps'    : float(crps_ensemble(y_true, q_arr).mean()),
        'cov90'   : float(np.mean((y_true >= lo90) & (y_true <= hi90))),
        'cov80'   : float(np.mean((y_true >= lo80) & (y_true <= hi80))),
        'n_total' : int(n),

        # detector
        'tp': tp, 'fp': fp, 'tn': tn, 'fn': fn,
        'accuracy' : float(accuracy),
        'precision': float(precision),
        'recall'   : float(recall),
        'f1'       : float(f1),
        'brier'    : brier,
    }

    # wet slice
    if obs_wet.sum() > 0:
        out.update({
            'rmse_wet'  : float(np.sqrt(np.mean((mean_mm[obs_wet] - y_true[obs_wet]) ** 2))),
            'mae_wet'   : float(np.mean(np.abs(mean_mm[obs_wet] - y_true[obs_wet]))),
            'crps_wet'  : float(crps_ensemble(y_true[obs_wet], q_arr[obs_wet]).mean()),
            'cov90_wet' : float(np.mean((y_true[obs_wet] >= lo90[obs_wet]) & (y_true[obs_wet] <= hi90[obs_wet]))),
            'cov80_wet' : float(np.mean((y_true[obs_wet] >= lo80[obs_wet]) & (y_true[obs_wet] <= hi80[obs_wet]))),
            'n_wet'     : int(obs_wet.sum()),
        })
    else:
        out.update({k: float('nan') for k in
                    ('rmse_wet','mae_wet','crps_wet','cov90_wet','cov80_wet')})
        out['n_wet'] = 0

    # dry slice
    if (~obs_wet).sum() > 0:
        d = ~obs_wet
        out.update({
            'mae_dry'  : float(np.mean(np.abs(mean_mm[d] - y_true[d]))),
            'rmse_dry' : float(np.sqrt(np.mean((mean_mm[d] - y_true[d]) ** 2))),
            'bias_dry' : float(np.mean(mean_mm[d] - y_true[d])),
            'n_dry'    : int(d.sum()),
        })
    else:
        out.update({'mae_dry': float('nan'), 'rmse_dry': float('nan'),
                    'bias_dry': float('nan'), 'n_dry': 0})
    return out


def _save_run(run_name: str, model, losses, preds_df, metrics, config):
    """Persist all artifacts under LOCAL_RUNS/<run_name>/.

    Known TFP issue: `model.params_` is a `structural_tuple.StructTuple`
    whose class is created dynamically inside a function (weakref cache),
    so `pickle` cannot find it by qualified name on reload. We flatten
    to a plain dict of named numpy leaves instead.
    """
    sub_dir = LOCAL_RUNS / run_name
    sub_dir.mkdir(parents=True, exist_ok=True)

    np.save(sub_dir / 'losses.npy', np.asarray(losses))
    with open(sub_dir / 'metrics.json', 'w') as f:
        json.dump(metrics, f, indent=2)
    with open(sub_dir / 'config.json', 'w') as f:
        json.dump(config, f, indent=2)
    preds_df.to_parquet(sub_dir / 'preds.parquet', index=False)

    # Flatten StructTuple into a plain dict.
    p = model.params_
    params_dict = {name: np.asarray(val) for name, val in zip(p._fields, p)}

    # Constructor kwargs so the model can be rebuilt on load.
    init_kwargs = {
        'width': model.width, 'depth': model.depth,
        'freq': model.freq, 'timetype': model.timetype,
        'seasonality_periods': list(model.seasonality_periods),
        'num_seasonal_harmonics': list(model.num_seasonal_harmonics),
        'feature_cols': list(model.feature_cols),
        'target_col':  model.target_col,
        'standardize': list(model.standardize),
        'observation_model': model.observation_model,
        'interactions': [list(t) for t in model.interactions]
                         if model.interactions is not None else None,
    }

    # data_handler scalers (mean / std for standardize) — needed at predict.
    dh = model.data_handler
    standardize_ = {
        'mean': np.asarray(dh.standardize_['mean'])  if hasattr(dh, 'standardize_') and dh.standardize_ else None,
        'std' : np.asarray(dh.standardize_['std'])   if hasattr(dh, 'standardize_') and dh.standardize_ else None,
    }

    payload = {
        'params_dict' : params_dict,
        'param_fields': list(p._fields),
        'init_kwargs' : init_kwargs,
        'standardize_': standardize_,
        'inference'   : config.get('inference'),
    }
    with open(sub_dir / 'model.pkl', 'wb') as f:
        pickle.dump(payload, f)
    print(f'  saved {len(params_dict)} param fields + meta to {sub_dir / "model.pkl"}')
    return sub_dir


def _load_run_model(run_name: str):
    """Inverse of _save_run: return an initialised estimator ready for predict.

    Not needed in this notebook (it reads preds.parquet directly), but
    useful for grid predictions later.
    """
    sub_dir = LOCAL_RUNS / run_name
    with open(sub_dir / 'model.pkl', 'rb') as f:
        payload = pickle.load(f)

    from tensorflow_probability.python.internal import structural_tuple
    inference = payload['inference']
    cls = bnf_st.BayesianNeuralFieldVI if inference == 'vi' else bnf_st.BayesianNeuralFieldMAP
    model = cls(**payload['init_kwargs'])

    # Rebuild StructTuple from the dict.
    StructT = structural_tuple.structtuple(payload['param_fields'])
    model.params_ = StructT(*(payload['params_dict'][k] for k in payload['param_fields']))

    # Restore scalers.
    std = payload.get('standardize_') or {}
    if std.get('mean') is not None and std.get('std') is not None:
        model.data_handler.standardize_ = {
            'mean': std['mean'], 'std': std['std'],
        }
    return model


def _upload_run_s3(run_name: str) -> None:
    """Idempotently upload LOCAL_RUNS/<run_name>/ to s3://bucket/bayesnf/runs/<run_name>/."""
    src_dir = LOCAL_RUNS / run_name
    if not src_dir.exists():
        print(f'  WARNING: {src_dir} does not exist, skip upload')
        return
    s3 = boto3.client('s3')
    for f in sorted(src_dir.iterdir()):
        if not f.is_file():
            continue
        key = f'{S3_RUNS_ROOT}/{run_name}/{f.name}'
        s3.upload_file(str(f), S3_BUCKET, key)
        size_mb = f.stat().st_size / 1e6
        print(f'  -> s3://{S3_BUCKET}/{key}  ({size_mb:.1f} MB)')


# ----------------------------------------------------------------------
def _build_predict_metrics(model, df_test_sub, name):
    """Run chunked predict, compute metrics, return (mean_per_point, q_arr, metrics_dict, preds_df, predict_time_s)."""
    means_chunks, q_chunks = [], []
    t0 = time.time()
    n = len(df_test_sub)
    for i in range(0, n, PRED_CHUNK):
        j = min(i + PRED_CHUNK, n)
        m, q = model.predict(df_test_sub.iloc[i:j], quantiles=QUANTILES, approximate_quantiles=True)
        means_chunks.append(np.asarray(m))
        q_chunks.append(np.asarray(q))
    gc.collect()
    predict_time_s = time.time() - t0
    print(f'  predict {n:,} rows in {predict_time_s:.0f}s')

    means_raw = np.concatenate(means_chunks, axis=-1)
    q_raw     = np.concatenate(q_chunks, axis=-1)
    mean_per_point = means_raw.reshape(-1, n).mean(axis=0)
    q_flat = q_raw.reshape(q_raw.shape[0], -1, n).mean(axis=1)

    if NEEDS_RESCALE:
        mean_per_point = mean_per_point / PRECIP_SCALE
        q_flat = q_flat / PRECIP_SCALE

    # --- diagnostics + safety clip ---------------------------------------
    n_bad_mean = int((~np.isfinite(mean_per_point)).sum())
    n_bad_q    = int((~np.isfinite(q_flat)).sum())
    abs_mean   = np.abs(mean_per_point)
    pct = np.nanpercentile(abs_mean, [50, 95, 99, 99.9])
    print(f'  diag |mean_mm|: p50={pct[0]:.2f}  p95={pct[1]:.2f}  '
          f'p99={pct[2]:.2f}  p99.9={pct[3]:.2f}  max={np.nanmax(abs_mean):.2f}')
    print(f'  diag non-finite: mean={n_bad_mean} ({n_bad_mean/n*100:.3f}%) '
          f'q={n_bad_q} ({n_bad_q/q_flat.size*100:.3f}%)')

    # Safety-net clip: 250 mm/day hard cap for station data
    # (world daily record ~ 1825 mm — extremely rare; in Germany
    # realistic max is <= 200 mm). The mean is clipped harder than the
    # quantiles to avoid breaking calibration.
    HARD_CAP_MEAN_MM = 250.0
    HARD_CAP_Q_MM    = 500.0
    n_clip_mean = int((mean_per_point > HARD_CAP_MEAN_MM).sum())
    n_clip_q    = int((q_flat > HARD_CAP_Q_MM).sum())
    if n_clip_mean or n_clip_q:
        print(f'  WARNING clipping: mean>{HARD_CAP_MEAN_MM}mm n={n_clip_mean}, '
              f'q>{HARD_CAP_Q_MM}mm n={n_clip_q}')
    mean_per_point = np.clip(np.nan_to_num(mean_per_point, nan=0.0,
                                           posinf=HARD_CAP_MEAN_MM, neginf=0.0),
                             0.0, HARD_CAP_MEAN_MM)
    q_flat = np.clip(np.nan_to_num(q_flat, nan=0.0,
                                    posinf=HARD_CAP_Q_MM, neginf=0.0),
                     0.0, HARD_CAP_Q_MM)

    y_true = df_test_sub['rainfall'].to_numpy()
    q_arr = np.stack(list(q_flat), axis=1)
    q_arr = np.sort(q_arr, axis=1)

    metrics = _metrics(y_true, mean_per_point, q_arr)
    metrics.update({
        'name': name, 'n_quantiles': len(QUANTILES),
        'predict_time_s': float(predict_time_s),
        'n_nonfinite_mean': n_bad_mean,
        'n_nonfinite_q'   : n_bad_q,
        'n_clipped_mean'  : n_clip_mean,
        'n_clipped_q'     : n_clip_q,
        'p99_abs_mean_mm' : float(pct[2]),
        'p999_abs_mean_mm': float(pct[3]),
    })

    preds_df = df_test_sub[['station_id', 'datetime']].copy()
    preds_df['observed_mm'] = y_true
    preds_df['mean_mm']     = mean_per_point
    for lbl, qv in zip(QUANTILE_LABELS, q_flat):
        preds_df[f'q{lbl:02d}'] = qv

    return mean_per_point, q_arr, metrics, preds_df


def _print_summary(metrics):
    m = metrics
    print(f'  CRPS     = {m["crps"]:.4f}    CRPS_wet = {m["crps_wet"]:.4f}')
    print(f'  MAE      = {m["mae"]:.4f}    MAE_wet  = {m["mae_wet"]:.4f}    MAE_dry  = {m["mae_dry"]:.4f}')
    print(f'  RMSE     = {m["rmse"]:.4f}   RMSE_wet = {m["rmse_wet"]:.4f}   RMSE_dry = {m["rmse_dry"]:.4f}')
    print(f'  bias     = {m["bias"]:+.4f}  cov90    = {m["cov90"]:.3f}      cov80    = {m["cov80"]:.3f}')
    print(f'  detector -> acc={m["accuracy"]:.3f}  P={m["precision"]:.3f}  R={m["recall"]:.3f}  F1={m["f1"]:.3f}  Brier={m["brier"]:.4f}')
    print(f'              TP={m["tp"]:,}  FP={m["fp"]:,}  TN={m["tn"]:,}  FN={m["fn"]:,}')


def _heartbeat_start(name):
    stop = threading.Event()
    def _beat():
        t0 = time.time()
        while not stop.wait(20.0):
            print(f'    [{name}] still training ({time.time()-t0:.0f}s)', flush=True)
    thr = threading.Thread(target=_beat, daemon=True); thr.start()
    return stop, thr


# ----------------------------------------------------------------------
def train_eval_vi(
    name: str,
    *,
    num_epochs: int,
    batch_size: int,
    ensemble_size: int,
    learning_rate: float,
    kl_weight: float,
    sample_size_posterior: int,
    width: int = 256,
    depth: int = 2,
    seed: int = 0,
    seasonality_periods=None,
    num_seasonal_harmonics=None,
    feature_cols=None,
    interactions=None,
) -> dict:
    seasonality_periods    = seasonality_periods    or SEAS_DEFAULT['periods']
    num_seasonal_harmonics = num_seasonal_harmonics or SEAS_DEFAULT['harmonics']
    feature_cols           = feature_cols           or GEO_DEFAULTS['feature_cols']
    interactions           = interactions           or GEO_DEFAULTS['interactions']

    assert len(seasonality_periods) == len(num_seasonal_harmonics)
    assert feature_cols[0] == 'datetime'
    for i, j in interactions:
        assert 0 <= i < len(feature_cols) and 0 <= j < len(feature_cols)

    print(f'\n{"="*78}')
    print(f'  [VI] {name}')
    print(f'       seasonality={seasonality_periods}/{num_seasonal_harmonics}')
    print(f'       features ({len(feature_cols)}): {feature_cols}')
    print(f'       width={width} depth={depth}  epochs={num_epochs}  batch={batch_size}')
    print(f'       ensemble={ensemble_size}  lr={learning_rate}  kl={kl_weight}  sample_post={sample_size_posterior}')
    print(f'{"="*78}')

    standardize_cols = [c for c in feature_cols if c != 'datetime']
    model = BayesianNeuralFieldVI(
        width=width, depth=depth,
        freq='D', timetype='index',
        seasonality_periods=seasonality_periods,
        num_seasonal_harmonics=num_seasonal_harmonics,
        feature_cols=feature_cols,
        target_col=TARGET_COL,
        standardize=standardize_cols,
        observation_model=LIKELIHOOD,
        interactions=interactions,
    )

    keep = feature_cols + ['rainfall', 'rainfall_int', 'station_id']
    df_train_sub = df_train[keep]
    df_test_sub  = df_test [keep]

    stop, thr = _heartbeat_start(name)
    t0 = time.time()
    try:
        model = model.fit(
            df_train_sub,
            seed=jax.random.PRNGKey(seed),
            num_epochs=num_epochs,
            ensemble_size=ensemble_size,
            learning_rate=learning_rate,
            batch_size=batch_size,
            kl_weight=kl_weight,
            sample_size_posterior=sample_size_posterior,
        )
    finally:
        stop.set(); thr.join(timeout=1)
    fit_time_s = time.time() - t0
    print(f'  fit done in {fit_time_s:.0f}s ({fit_time_s/60:.1f} min)')

    losses_arr = np.asarray(model.losses_)
    _, _, metrics, preds_df = _build_predict_metrics(model, df_test_sub, name)
    metrics.update({
        'inference'             : 'vi',
        'seasonality_periods'   : seasonality_periods,
        'num_seasonal_harmonics': num_seasonal_harmonics,
        'feature_cols'          : feature_cols,
        'interactions'          : interactions,
        'n_features'            : len(feature_cols),
        'fit_time_s'            : float(fit_time_s),
    })

    config = {
        'inference': 'vi', 'name': name,
        'width': width, 'depth': depth,
        'num_epochs': num_epochs, 'batch_size': batch_size,
        'ensemble_size': ensemble_size, 'learning_rate': learning_rate,
        'kl_weight': kl_weight, 'sample_size_posterior': sample_size_posterior,
        'seed': seed,
        'seasonality_periods': seasonality_periods,
        'num_seasonal_harmonics': num_seasonal_harmonics,
        'feature_cols': feature_cols, 'interactions': interactions,
        'likelihood': LIKELIHOOD, 'target_col': TARGET_COL,
        'fold': FOLD, 'year_start': YEAR_START, 'year_end': YEAR_END,
    }
    _save_run(name, model, losses_arr, preds_df, metrics, config)
    _upload_run_s3(name)
    _print_summary(metrics)

    del model
    jax.clear_caches(); gc.collect()
    return metrics


def train_eval_map(
    name: str,
    *,
    num_epochs: int,
    batch_size: int,
    ensemble_size: int,
    learning_rate: float,
    num_splits: int = 1,
    width: int = 256,
    depth: int = 2,
    seed: int = 0,
    seasonality_periods=None,
    num_seasonal_harmonics=None,
    feature_cols=None,
    interactions=None,
) -> dict:
    seasonality_periods    = seasonality_periods    or SEAS_DEFAULT['periods']
    num_seasonal_harmonics = num_seasonal_harmonics or SEAS_DEFAULT['harmonics']
    feature_cols           = feature_cols           or GEO_DEFAULTS['feature_cols']
    interactions           = interactions           or GEO_DEFAULTS['interactions']

    assert len(seasonality_periods) == len(num_seasonal_harmonics)
    assert feature_cols[0] == 'datetime'

    print(f'\n{"="*78}')
    print(f'  [MAP] {name}')
    print(f'        seasonality={seasonality_periods}/{num_seasonal_harmonics}')
    print(f'        features ({len(feature_cols)}): {feature_cols}')
    print(f'        width={width} depth={depth}  epochs={num_epochs}  batch={batch_size}')
    print(f'        ensemble={ensemble_size}  lr={learning_rate}  splits={num_splits}')
    print(f'{"="*78}')

    standardize_cols = [c for c in feature_cols if c != 'datetime']
    model = BayesianNeuralFieldMAP(
        width=width, depth=depth,
        freq='D', timetype='index',
        seasonality_periods=seasonality_periods,
        num_seasonal_harmonics=num_seasonal_harmonics,
        feature_cols=feature_cols,
        target_col=TARGET_COL,
        standardize=standardize_cols,
        observation_model=LIKELIHOOD,
        interactions=interactions,
    )

    keep = feature_cols + ['rainfall', 'rainfall_int', 'station_id']
    df_train_sub = df_train[keep]
    df_test_sub  = df_test [keep]

    stop, thr = _heartbeat_start(name)
    t0 = time.time()
    try:
        model = model.fit(
            df_train_sub,
            seed=jax.random.PRNGKey(seed),
            num_epochs=num_epochs,
            ensemble_size=ensemble_size,
            learning_rate=learning_rate,
            batch_size=batch_size,
            num_splits=num_splits,
        )
    finally:
        stop.set(); thr.join(timeout=1)
    fit_time_s = time.time() - t0
    print(f'  fit done in {fit_time_s:.0f}s ({fit_time_s/60:.1f} min)')

    losses_arr = np.asarray(model.losses_)
    _, _, metrics, preds_df = _build_predict_metrics(model, df_test_sub, name)
    metrics.update({
        'inference'             : 'map',
        'seasonality_periods'   : seasonality_periods,
        'num_seasonal_harmonics': num_seasonal_harmonics,
        'feature_cols'          : feature_cols,
        'interactions'          : interactions,
        'n_features'            : len(feature_cols),
        'fit_time_s'            : float(fit_time_s),
    })

    config = {
        'inference': 'map', 'name': name,
        'width': width, 'depth': depth,
        'num_epochs': num_epochs, 'batch_size': batch_size,
        'ensemble_size': ensemble_size, 'learning_rate': learning_rate,
        'num_splits': num_splits, 'seed': seed,
        'seasonality_periods': seasonality_periods,
        'num_seasonal_harmonics': num_seasonal_harmonics,
        'feature_cols': feature_cols, 'interactions': interactions,
        'likelihood': LIKELIHOOD, 'target_col': TARGET_COL,
        'fold': FOLD, 'year_start': YEAR_START, 'year_end': YEAR_END,
    }
    _save_run(name, model, losses_arr, preds_df, metrics, config)
    _upload_run_s3(name)
    _print_summary(metrics)

    del model
    jax.clear_caches(); gc.collect()
    return metrics


print('helpers ready')
print('  GEO_DEFAULTS : n_features =', len(GEO_DEFAULTS['feature_cols']),
      ', n_interactions =', len(GEO_DEFAULTS['interactions']))
print('  SEAS_DEFAULT :', SEAS_DEFAULT)
print('  thresholds   : observed wet >=', WET_THRESHOLD_MM, 'mm,  predicted wet >=', PRED_DRY_THRESHOLD, 'mm')


## 7. Train (VI)


In [ ]:
m_vi_final = train_eval_vi(
    name='vi__final__WY_h1_10__ffrk_full',
    num_epochs            = 100,
    batch_size            = 131072,
    ensemble_size         = 16,
    learning_rate         = 0.005,
    kl_weight             = 0.1,
    sample_size_posterior = 5,
    width = 256,
    depth = 2,
    seed  = 0,
)

print('\n' + '='*78)
print('  final result (VI, train -> 492 holdout stations)')
print('='*78)
print(f'  CRPS     : {m_vi_final["crps"]:.4f}    CRPS_wet : {m_vi_final["crps_wet"]:.4f}')
print(f'  MAE      : {m_vi_final["mae"]:.4f}    MAE_wet  : {m_vi_final["mae_wet"]:.4f}')
print(f'  RMSE     : {m_vi_final["rmse"]:.4f}   RMSE_wet : {m_vi_final["rmse_wet"]:.4f}')
print(f'  bias     : {m_vi_final["bias"]:+.4f}')
print(f'  cov90/80 : {m_vi_final["cov90"]:.3f} / {m_vi_final["cov80"]:.3f}')
print(f'  n_test   : {m_vi_final["n_total"]:,}  (wet: {m_vi_final["n_wet"]:,})')
